In [1]:
"""
Универсальный пайплайн детекции R/P/T-зубцов — работает с ЛЮБЫМ EDF-файлом
=============================================================================

"""
!pip install pyEDFlib PyWavelets -q
import math
import warnings
import numpy as np
import pyedflib
from scipy.signal import find_peaks, cheby2, filtfilt


# ══════════════════════════════════════════════════════════════════════
# 1. Универсальное чтение EDF (авто-канал, авто-масштаб единиц)
# ══════════════════════════════════════════════════════════════════════

def find_ecg_channel(labels, preferred=None):
    """Находит индекс ЭКГ-канала по названию. Если preferred задан и
    найден — берёт его; иначе первый канал, содержащий 'ECG'/'ЭКГ'."""
    if preferred is not None:
        for i, lab in enumerate(labels):
            if preferred.lower() in lab.lower():
                return i
    for i, lab in enumerate(labels):
        if "ecg" in lab.lower() or "экг" in lab.lower():
            return i
    return 0  # запасной вариант — первый канал


_UNIT_TO_MV = {
    "v": 1000.0, "volt": 1000.0, "volts": 1000.0,
    "mv": 1.0, "millivolt": 1.0, "millivolts": 1.0,
    "uv": 0.001, "µv": 0.001, "microvolt": 0.001, "microvolts": 0.001,
}


def read_edf_channel_mV(path: str, channel_name: str = None):
    """
    Универсальное чтение одного канала EDF в мВ.

    Автоматически:
      - находит ЭКГ-канал по названию (если channel_name не задан явно
        или не найден — берёт первый похожий на ЭКГ);
      - приводит физические единицы к мВ по метаданным заголовка файла
        (а не по жёстко зашитому предположению).
    """
    f = pyedflib.EdfReader(path)
    try:
        labels = f.getSignalLabels()
        ch = find_ecg_channel(labels, channel_name)
        fs = float(f.getSampleFrequency(ch))
        sig = f.readSignal(ch).astype(np.float64)
        unit = f.getPhysicalDimension(ch).strip().lower()
        scale = _UNIT_TO_MV.get(unit)
        if scale is None:
            warnings.warn(f"Неизвестная единица измерения '{unit}' — "
                           f"сигнал возвращён БЕЗ масштабирования, проверьте вручную!")
            scale = 1.0
        sig_mV = sig * scale
        return sig_mV, fs, labels[ch]
    finally:
        f.close()


# ══════════════════════════════════════════════════════════════════════
# 2. Универсальный (адаптивный) детектор R-зубцов
# ══════════════════════════════════════════════════════════════════════

def _sine_wavelet(length: int) -> np.ndarray:
    t = np.linspace(0, math.pi, length)
    w = np.sin(t)
    return w - w.mean()


def filter_ecg(sig_mV: np.ndarray, fs: float, fc: float = 0.5, order: int = 5):
    """Чебышёвский ФВЧ (тип II, rs=40дБ): убирает дрейф изолинии перед
    детекцией R. Частота среза fc и порядок универсальны (не зависят от
    конкретного файла) — работают на любой физиологической ЭКГ."""
    nyq = fs / 2.0
    b, a = cheby2(order, rs=40, Wn=fc / nyq, btype="high", analog=False)
    ecg_hf = filtfilt(b, a, sig_mV)
    return ecg_hf


def _wavelet_image(sig: np.ndarray, w: np.ndarray) -> np.ndarray:
    L, N, half = len(w), len(sig), len(w) // 2
    out = np.zeros(N)
    for i in range(half, N - half):
        out[i] = float(np.dot(sig[i - half:i - half + L], w))
    return out


def detect_r_peaks(ecg_mV: np.ndarray, fs: float, min_rr_ms: float = 300,
                    k_threshold: float = 2.0) -> np.ndarray:
    """
    Детекция R-зубцов с АДАПТИВНЫМ порогом (mean + k*std от вейвлет-образа
    сигнала), а не фиксированным числом — порог пересчитывается заново
    для каждого сигнала, поэтому корректно работает при любой амплитуде
    записи. Возвращает индексы отсчётов (не время в секундах).
    """
    wlen = max(3, int(0.12 * fs))
    w = _sine_wavelet(wlen)
    wav = _wavelet_image(ecg_mV, w)
    level = wav.mean() + k_threshold * wav.std()
    above = np.where(wav > level)[0]
    if len(above) == 0:
        return np.array([], dtype=int)

    min_gap = int(min_rr_ms / 1000.0 * fs)
    peaks, group = [], [above[0]]
    for idx in above[1:]:
        if idx - group[-1] <= 1:
            group.append(idx)
        else:
            peaks.append(group[0] + int(np.argmax(wav[group])))
            group = [idx]
    peaks.append(group[0] + int(np.argmax(wav[group])))

    out = [peaks[0]]
    for p in peaks[1:]:
        if p - out[-1] >= min_gap:
            out.append(p)
    return np.array(out, dtype=int)


def check_double_counting(r_idx: np.ndarray, fs: float, tol: float = 0.15) -> bool:
    """
    Самопроверка на задвоение: если RR-интервалы регулярно чередуются
    короткий/длинный (и их сумма примерно постоянна), это признак того,
    что детектор ловит два срабатывания на один удар. Возвращает True,
    если похоже на задвоение (стоит проверить параметры вручную).
    """
    if len(r_idx) < 5:
        return False
    rr = np.diff(r_idx) / fs
    short = rr[0::2]
    long_ = rr[1::2]
    n = min(len(short), len(long_))
    if n < 2:
        return False
    pair_sum = short[:n] + long_[:n]
    is_alternating = (np.std(short[:n]) / np.mean(short[:n]) < tol and
                       np.std(long_[:n]) / np.mean(long_[:n]) < tol and
                       np.mean(short[:n]) < 0.6 * np.mean(long_[:n]))
    return bool(is_alternating)


# ══════════════════════════════════════════════════════════════════════
# 3. filter_qrs / detect_pt_peaks — уже адаптивные
# ══════════════════════════════════════════════════════════════════════

def filter_qrs(ecg_mV: np.ndarray, r_idx: np.ndarray, fs: float) -> np.ndarray:
    """Вырезает QRS-комплекс линейной интерполяцией в окне [-50мс,+100мс]."""
    half, qrs_e = int(0.05 * fs), int(0.10 * fs)
    out = ecg_mV.copy()
    for r in r_idx:
        i0, i1 = max(0, r - half), min(len(ecg_mV) - 1, r + qrs_e)
        if i1 - i0 < 2:
            continue
        x = np.arange(i0, i1 + 1)
        out[i0:i1 + 1] = np.interp(x, [i0, i1], [ecg_mV[i0], ecg_mV[i1]])
    return out


def _halfwave_wavelet(lam: int = 17) -> np.ndarray:
    if lam % 2 == 0:
        lam += 1
    t = np.linspace(0, math.pi, lam)
    pos = np.sin(t)
    area = float(np.trapezoid(pos)) if hasattr(np, "trapezoid") else float(np.trapz(pos))
    wing = max(1, lam // 4)
    w = np.zeros(lam + 2 * wing)
    w[:wing] = -area / (2 * wing)
    w[wing:wing + lam] = pos
    w[wing + lam:] = -area / (2 * wing)
    return w - w.mean()


def detect_pt_peaks(ecg_f: np.ndarray, r_idx: np.ndarray, fs: float, lam: int = 17):
    """P/T-зубцы: окна поиска — доли ФАКТИЧЕСКОГО соседнего RR (не
    фиксированные мс), кандидат — максимум вейвлет-согласованного
    отклика выше порога.

    Порог = 0.3 * (максимум вейвлет-отклика, ИСКЛЮЧАЯ участки внутри
    аномально длинных RR-интервалов, RR > 1.8*медианы). Такие участки —
    признак пропущенного R-зубца: его QRS-комплекс не вырезан
    интерполяцией (filter_qrs интерполирует только вокруг НАЙДЕННЫХ R) и
    даёт в вейвлет-отклике выброс, в разы превышающий типичный P/T-сигнал.
    Без этого исключения единственный такой выброс на коротком фрагменте
    может задрать порог настолько, что почти все настоящие P/T окажутся
    ниже него (проверено: на 10-секундном окне с одним пропущенным
    ударом старый вариант находил 1 P и 1 T вместо ожидаемых ~17)."""
    w = _halfwave_wavelet(lam)
    wav = _wavelet_image(ecg_f, w)

    mask = np.ones(len(wav), dtype=bool)
    if len(r_idx) > 2:
        rr = np.diff(r_idx)
        med_rr = np.median(rr)
        if med_rr > 0:
            for k in range(len(rr)):
                if rr[k] > 1.8 * med_rr:
                    mask[r_idx[k]:r_idx[k + 1]] = False
    pos = wav[(wav > 0) & mask]
    if len(pos) == 0:
        pos = wav[wav > 0]
    thr = 0.3 * pos.max() if len(pos) else 0
    p_idx, t_idx = [], []
    for k in range(len(r_idx)):
        r = r_idx[k]
        r_prev = r_idx[k - 1] if k > 0 else max(0, r - int(fs))
        r_next = r_idx[k + 1] if k < len(r_idx) - 1 else min(len(ecg_f) - 1, r + int(fs))

        rr = r - r_prev
        ps, pe = r_prev + max(1, rr // 6), r - max(1, rr // 6)
        if ps < pe:
            seg = wav[ps:pe]
            if seg.max() > thr:
                p_idx.append(ps + int(np.argmax(seg)))

        rnn = r_next - r
        ts, te = r + max(1, rnn // 5), r + min(rnn - 1, int(0.75 * rnn))
        if ts < te <= len(wav):
            seg = wav[ts:te]
            if len(seg) > 0 and seg.max() > thr:
                t_idx.append(ts + int(np.argmax(seg)))
    return np.array(p_idx, dtype=int), np.array(t_idx, dtype=int)


# ══════════════════════════════════════════════════════════════════════
# 4. Готовая функция "на вход путь к EDF — на выход R/P/T"
# ══════════════════════════════════════════════════════════════════════

def analyze_edf(path: str, channel_name: str = None, window_s: tuple = None,
                 verbose: bool = True):
    """
    Полный универсальный анализ одного файла EDF:
    читает канал -> детектирует R (адаптивный порог) -> вырезает QRS ->
    детектирует P/T -> проверяет на задвоение.

    window_s: (t0, t1) в секундах — если None, берётся весь файл.
    Возвращает dict с сигналом, fs, индексами R/P/T и флагом задвоения.
    """
    sig_mV, fs, used_label = read_edf_channel_mV(path, channel_name)

    if window_s is not None:
        i0, i1 = int(window_s[0] * fs), int(window_s[1] * fs)
        sig_mV = sig_mV[i0:i1]
    else:
        i0 = 0

    ecg_hf = filter_ecg(sig_mV, fs)
    r_idx = detect_r_peaks(ecg_hf, fs)
    doubled = check_double_counting(r_idx, fs)
    if doubled and verbose:
        warnings.warn(f"[{path}] Похоже на задвоение R-пиков (чередующиеся "
                       f"RR-интервалы) — проверьте сигнал вручную.")

    ecg_f = filter_qrs(ecg_hf, r_idx, fs)
    p_idx, t_idx = detect_pt_peaks(ecg_f, r_idx, fs)

    if verbose:
        dur_s = len(sig_mV) / fs
        hr = 60.0 / np.mean(np.diff(r_idx) / fs) if len(r_idx) > 1 else float("nan")
        print(f"[{path}] канал={used_label} fs={fs:.0f}Гц окно={dur_s:.1f}с | "
              f"R={len(r_idx)} P={len(p_idx)} T={len(t_idx)} | "
              f"ЧСС≈{hr:.1f} уд/мин | задвоение={'ДА !!!' if doubled else 'нет'}")

    return {
        "signal_mV": sig_mV, "fs": fs, "channel": used_label,
        "r_idx": r_idx, "p_idx": p_idx, "t_idx": t_idx,
        "ecg_qrs_removed": ecg_f, "offset_samples": i0,
        "double_counting_suspected": doubled,
    }


def plot_analysis(path: str, res_full: dict, res_win: dict, window_s: tuple,
                   save_path: str = None):
    """Строит рисунок ПО ТЕМ ЖЕ данным, что были напечатаны в
    консоль (res_full/res_win — результаты analyze_edf), поэтому числа в
    заголовке рисунка и числа, выведенные print()-ом, гарантированно
    совпадают — это один и тот же вызов, не два разных окна в разных
    местах кода."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fs = res_full["fs"]
    r_idx_full = res_full["r_idx"]
    rr_s = np.diff(r_idx_full) / fs
    t_mid = (r_idx_full[:-1] + r_idx_full[1:]) / 2 / fs / 60
    hr = 60.0 / rr_s

    sig_raw = res_win["signal_mV"]
    sig_qrs_removed = res_win["ecg_qrs_removed"]
    fs_w = res_win["fs"]
    r_w, p_w, t_w = res_win["r_idx"], res_win["p_idx"], res_win["t_idx"]
    t_axis = np.arange(len(sig_raw)) / fs_w

    fig = plt.figure(figsize=(14, 11))
    gs = fig.add_gridspec(3, 1, height_ratios=[1.3, 1, 1])

    ax1 = fig.add_subplot(gs[0])
    ax1.plot(t_axis, sig_raw, lw=0.8, color="tab:blue", label="ЭКГ (после ФВЧ)")
    ax1.scatter(r_w / fs_w, sig_raw[r_w], color="red", s=60, zorder=5, label=f"R (n={len(r_w)})")
    if len(p_w):
        ax1.scatter(p_w / fs_w, sig_qrs_removed[p_w], color="green", marker="^", s=55, zorder=5, label=f"P (n={len(p_w)})")
    if len(t_w):
        ax1.scatter(t_w / fs_w, sig_qrs_removed[t_w], color="blue", marker="v", s=55, zorder=5, label=f"T (n={len(t_w)})")
    ax1.set_title(f"{path} — канал {res_win['channel']}, фрагмент {window_s[0]}–{window_s[1]} с "
                  f"| R={len(r_w)} P={len(p_w)} T={len(t_w)}")
    ax1.set_xlabel("Время, с"); ax1.set_ylabel("мВ")
    ax1.legend(loc="upper right", fontsize=8); ax1.grid(alpha=0.3)

    ax2 = fig.add_subplot(gs[1])
    ax2.plot(rr_s, lw=0.7, color="tab:orange")
    ax2.scatter(np.arange(len(rr_s)), rr_s, s=4, color="tab:orange")
    ax2.axhline(np.median(rr_s), color="gray", ls="--", lw=1, label=f"медиана={np.median(rr_s):.2f}с")
    ax2.set_title(f"RR-интервалы по всей записи (n={len(rr_s)}) — "
                  f"{'ЗАДВОЕНИЕ ЗАПОДОЗРЕНО' if res_full['double_counting_suspected'] else 'чередования короткий/длинный нет'}")
    ax2.set_xlabel("Номер удара"); ax2.set_ylabel("RR, с")
    ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

    ax3 = fig.add_subplot(gs[2])
    ax3.plot(t_mid, hr, lw=0.5, color="tab:purple")
    ax3.set_title(f"Динамика ЧСС по всей записи (R={len(r_idx_full)}, "
                  f"P={len(res_full['p_idx'])}, T={len(res_full['t_idx'])}, "
                  f"медиана ЧСС={np.median(hr):.1f} уд/мин)")
    ax3.set_xlabel("Время, мин"); ax3.set_ylabel("ЧСС, уд/мин")
    ax3.grid(alpha=0.3)

    fig.suptitle(f"{path} — детекция R/P/T универсальным адаптивным пайплайном", fontsize=13, y=1.0)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=140, bbox_inches="tight")
        plt.close(fig)
        print(f"  -> сохранено: {save_path}")
    return fig


def run_full_report(path: str, window_s: tuple = (0, 10), save_dir: str = "."):
    """Единая точка входа: считает ПОЛНУЮ запись (для RR/ЧСС по всей
    длительности и для проверки на задвоение) и ОДНО окно window_s (для
    наглядной картинки R/P/T) — и печатает, и строит рисунок СТРОГО по
    этим же двум результатам, без повторных независимых вызовов с
    другими параметрами где-то ещё в коде."""
    print(f"\n=== {path} ===")
    print(f"Полная запись:")
    res_full = analyze_edf(path, verbose=True)
    print(f"Окно {window_s[0]}-{window_s[1]}с (то же самое показано на рисунке):")
    res_win = analyze_edf(path, window_s=window_s, verbose=True)
    save_path = f"{save_dir}/universal_{path.replace('.edf','')}.png"
    plot_analysis(path, res_full, res_win, window_s, save_path=save_path)
    return res_full, res_win


if __name__ == "__main__":
    import glob
    edf_files = sorted(glob.glob("*.edf"))
    print(f"Найдено файлов EDF: {edf_files}")
    for path in edf_files:
        run_full_report(path, window_s=(0, 10))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 32.5 MB/s eta 0:00:00
Найдено файлов EDF: ['01_ГУСА.edf', '07_ВОРО.edf', '10_МИТИ.edf', '30_КРУЧ.edf']

=== 01_ГУСА.edf ===
Полная запись:
[01_ГУСА.edf] канал=ECG V2-Ref fs=200Гц окно=3330.0с | R=3634 P=1501 T=1499 | ЧСС≈65.8 уд/мин | задвоение=нет
Окно 0-10с (то же самое показано на рисунке):
[01_ГУСА.edf] канал=ECG V2-Ref fs=200Гц окно=10.0с | R=17 P=17 T=17 | ЧСС≈103.4 уд/мин | задвоение=нет
  -> сохранено: ./universal_01_ГУСА.png

=== 07_ВОРО.edf ===
Полная запись:
[07_ВОРО.edf] канал=ECG V2-Ref fs=200Гц окно=3180.0с | R=2274 P=1 T=1 | ЧСС≈42.9 уд/мин | задвоение=нет
Окно 0-10с (то же самое показано на рисунке):
[07_ВОРО.edf] канал=ECG V2-Ref fs=200Гц окно=10.0с | R=2 P=2 T=1 | ЧСС≈144.6 уд/мин | задвоение=нет
  -> сохранено: ./universal_07_ВОРО.png

=== 10_МИТИ.edf ===
Полная запись:
[10_МИТИ.edf] канал=ECG V2-Ref fs=200Гц окно=3450.0с | R=2954 P=58 T=53 | ЧСС≈51.4 уд/мин | задвоение=нет
Окно 0-10с (то же самое показано на рис